In [1]:
import csv
import os
import requests

def download_pdfs_from_csv(csv_url: str, output_dir: str = "./pdfs") -> None:

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Download and parse the CSV from the URL
    response = requests.get(csv_url)
    response.raise_for_status()

    for row in csv.DictReader(response.text.splitlines()):
        title = row["title"]
        pdf_url = row["pdf_url"]

        # Skip rows with no PDF URL
        if not pdf_url:
            print(f"Skipping '{title}' — no PDF URL")
            continue

        # Sanitize the title to use as a safe filename
        safe_title = "".join(c if c.isalnum() or c in " -_" else "_" for c in title).strip()
        output_path = os.path.join(output_dir, f"{safe_title}.pdf")

        print(f"Downloading: {title}")
        try:
            # Stream the PDF to avoid loading the entire file into memory
            pdf_response = requests.get(pdf_url, stream=True)
            pdf_response.raise_for_status()
            # Write the PDF in chunks to the output file
            with open(output_path, "wb") as f:
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  Saved to {output_path}")
        except requests.RequestException as e:
            print(f"  Error: {e}")

In [2]:
download_pdfs_from_csv("https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv", "books")

Downloading: Think Python 2e
  Saved to books/Think Python 2e.pdf
Downloading: Think DSP
  Saved to books/Think DSP.pdf
Downloading: Think Complexity 2e
  Saved to books/Think Complexity 2e.pdf
Downloading: Think Java 2e
  Saved to books/Think Java 2e.pdf
Downloading: Physical Modeling in MATLAB
  Saved to books/Physical Modeling in MATLAB.pdf
Downloading: Think OS
  Saved to books/Think OS.pdf
Downloading: Think C++
  Saved to books/Think C__.pdf


In [3]:
from markitdown import MarkItDown

def convert_pdfs_to_markdown(books_dir: str = "books", output_dir: str = "books-md") -> None:
    os.makedirs(output_dir, exist_ok=True)
    md = MarkItDown()

    for filename in os.listdir(books_dir):
        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(books_dir, filename)
        book_name = os.path.splitext(filename)[0]
        output_path = os.path.join(output_dir, f"{book_name}.md")

        print(f"Converting: {filename}")
        try:
            result = md.convert(pdf_path)
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(result.text_content)
            print(f"  Saved to {output_path} ({len(result.text_content):,} chars)")
        except Exception as e:
            print(f"  Error: {e}")


convert_pdfs_to_markdown("books", "books-text")

Converting: Think OS.pdf
  Saved to books-text/Think OS.md (233,518 chars)
Converting: Think Java 2e.pdf
  Saved to books-text/Think Java 2e.md (911,790 chars)
Converting: Physical Modeling in MATLAB.pdf
  Saved to books-text/Physical Modeling in MATLAB.md (391,829 chars)
Converting: Think Python 2e.pdf
  Saved to books-text/Think Python 2e.md (450,161 chars)
Converting: Think DSP.pdf
  Saved to books-text/Think DSP.md (356,248 chars)
Converting: Think Complexity 2e.pdf
  Saved to books-text/Think Complexity 2e.md (503,183 chars)
Converting: Think C__.pdf
  Saved to books-text/Think C__.md (349,497 chars)


In [6]:
def load_markdown_documents(folder: str) -> list[dict]:
    documents = []

    for filename in os.listdir(folder):
        if not filename.lower().endswith(".md"):
            continue

        filepath = os.path.join(folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()

        content = [line.strip() for line in lines if line.strip()]
        documents.append({"source": filename, "content": content})

    return documents


documents = load_markdown_documents("books-text")
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  {doc['source']}: {len(doc['content'])} lines")

Loaded 7 documents
  Think Java 2e.md: 12373 lines
  Physical Modeling in MATLAB.md: 5978 lines
  Think DSP.md: 5524 lines
  Think Python 2e.md: 10709 lines
  Think OS.md: 3444 lines
  Think Complexity 2e.md: 7216 lines
  Think C__.md: 5371 lines


In [7]:
## Chunking

from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=100, step=50)

In [8]:
## Question 2
## How many chunks are produced for the "Think Python" book with these settings?

think_python_chunks = [c for c in document_chunks if "Think Python" in c["source"]]    
print(len(think_python_chunks))     

214


In [9]:
document_chunks[9]

{'start': 450,
 'content': ['Hello.class. To run the program, the programmer uses java to interpret the',
  'byte code. The result of the program is then displayed on the screen.',
  'Although it might seem complicated, these steps are automated for you in',
  'most development environments. Usually, you only have to press a button or',
  'type a single command to compile and interpret your program. On the other',
  'hand, it is important to know what steps are happening in the background, so',
  '| if something | goes       | wrong | you | can figure | out | what it is. |     |',
  '| ------------ | ---------- | ----- | --- | ---------- | --- | ----------- | --- |',
  '| 1.5          | Displaying |       | Two | Messages   |     |             |     |',
  'You can put as many statements as you like in the main method. For example,',
  '| to display | more than | one | line | of output: |     |     |     |',
  '| ---------- | --------- | --- | ---- | ---------- | --- | --- | --- |',
  '

In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["source"]
)

chunks_for_index = []
for chunk in document_chunks:
    chunks_for_index.append({
        "source": chunk["source"],
        "content": "\n".join(chunk["content"])
    })

index.fit(chunks_for_index)

In [12]:
## Question 3: How many documents (chunks) did you index?

len(chunks_for_index)

1009

In [13]:
results = index.search("python function definition", num_results=5)

In [14]:
results

[{'source': 'Think Python 2e.md',
  'content': 'when you are comfortable with Python, I’ll make suggestions for installing Python on your\ncomputer.\nThere are a number of web pages you can use to run Python. If you already have a fa-\nvorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed\ninstructions for getting started at http://tinyurl.com/thinkpython2e.\nThere are two versions of Python, called Python 2 and Python 3. They are very similar, so\nif you learn one, it is easy to switch to the other. In fact, there are only a few differences you\nwill encounter as a beginner. This book is written for Python 3, but I include some notes\nabout Python 2.\nThe Python interpreter is a program that reads and executes Python code. Depending\non your environment, you might start the interpreter by clicking on an icon, or by typing\npython on a command line. When it starts, you should see output like this:\nPython 3.4.0 (default, Jun 19 2015, 14:20:21)\n[GCC 4.8.

In [ ]:
## Question 4: Top Result - Think Python 2e

## RAG

In [37]:
from openai import OpenAI
openai_client = OpenAI()

import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    return response

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [38]:
answer = rag("python function definition")

In [39]:
answer

Response(id='resp_0846e4371a1803100069eddd61139481979eccb43900f5c738', created_at=1777196385.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0846e4371a1803100069eddd627474819795f182f6f1f6eee7', content=[ResponseOutputText(annotations=[], text="In Python, a function is defined using the `def` keyword, followed by the function name and parentheses enclosing any parameters. The body of the function is indented and contains the statements that define what the function does. Here's the general syntax for defining a function:\n\n```python\ndef function_name(parameters):\n    # body of the function\n    # perform operations\n    return result  # optional return statement\n```\n\nFor example, to define a simple function that adds two numbers:\n\n```python\ndef add(a, b):\n    return a + b\n```\n\nYou can then call this function using its name and passing the required arguments:\n\n`

In [40]:
answer.output_text

"In Python, a function is defined using the `def` keyword, followed by the function name and parentheses enclosing any parameters. The body of the function is indented and contains the statements that define what the function does. Here's the general syntax for defining a function:\n\n```python\ndef function_name(parameters):\n    # body of the function\n    # perform operations\n    return result  # optional return statement\n```\n\nFor example, to define a simple function that adds two numbers:\n\n```python\ndef add(a, b):\n    return a + b\n```\n\nYou can then call this function using its name and passing the required arguments:\n\n```python\nresult = add(5, 3)  # This will return 8\n```\n\nMake sure to follow the indentation rules of Python when defining the body of your function, as this is essential for the function to work properly."

In [41]:
##Question 5: How many input tokens did we use for this one RAG query?

answer.usage.input_tokens

7218

## Structured vs Unstructured Output

In [28]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [29]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'Suggested follow-up questions',
   'items': {'type': 'string'},
   'title': 'Followup Questions',
   'type': 'array'}},
 'required': ['answer',
  'found_answer',
  'confidence',
  'confidence_expla

In [35]:
# structured

def llm(user_prompt, instructions, output_type, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )
    return response

def rag(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions, output_type)
    return answer

In [36]:
answer_structured = rag("python function definition")

In [42]:
## Question 6: How many more input tokens does the structured output version
## used compared to the unstructured version

answer_structured.usage.input_tokens 

7404

In [33]:
## 7404 - 7218 = 186, structured query has 186 more input tokens